# Module 4 — SOLUTION: RAG Evaluation with RAGAS

> **Instructor note:** This is the complete solution. Share after the exercise session.

In [ ]:
import sys
sys.path.insert(0, ".")  # solutions/src/ takes priority

import json
import os
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI
import chromadb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

load_dotenv("../.env")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_or_create_collection(
    name="workshop_rag", metadata={"hnsw:space": "cosine"}
)
llm_client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.com/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)
FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")
SMART_MODEL = os.getenv("NETLIGHT_MODEL_SMART", "claude-sonnet-4-5")

print(f"Collection: {collection.count()} docs")

In [ ]:
# RAGPipeline (same as Module 3 solution)
SYSTEM_PROMPT = """\
You are a knowledgeable insurance and technology assistant.
Answer ONLY using information from the numbered context chunks provided.
Cite sources using the chunk number, e.g. "According to [1]..."
If the context is insufficient, say so explicitly.
Keep answers concise. Never fabricate facts."""


def build_prompt(question: str, chunks: list[dict]) -> str:
    lines = []
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"[{i}] Source: {chunk['source']} (relevance: {chunk.get('similarity', 0):.2f})")
        lines.append(chunk["text"])
        lines.append("")
    return f"Retrieved Context:\n{''.join(lines)}\nQuestion: {question}\n\nAnswer:"


class RAGPipeline:
    def __init__(self, collection, embed_model, llm_client, model=FAST_MODEL,
                 top_k=5, system_prompt=SYSTEM_PROMPT, min_similarity=0.0):
        self.collection = collection
        self.embed_model = embed_model
        self.llm_client = llm_client
        self.model = model
        self.top_k = top_k
        self.system_prompt = system_prompt
        self.min_similarity = min_similarity

    def retrieve(self, question: str) -> list[dict]:
        q_emb = self.embed_model.encode(question).tolist()
        results = self.collection.query(
            query_embeddings=[q_emb], n_results=self.top_k,
            include=["documents", "metadatas", "distances"]
        )
        chunks = [
            {"text": t, "source": m["source"], "similarity": 1 - d}
            for t, m, d in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
        ]
        return [c for c in chunks if c["similarity"] >= self.min_similarity]

    def ask(self, question: str) -> dict:
        chunks = self.retrieve(question)
        prompt = build_prompt(question, chunks)
        response = self.llm_client.chat.completions.create(
            model=self.model,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ],
        )
        return {
            "question": question,
            "answer": response.choices[0].message.content,
            "contexts": [c["text"] for c in chunks],
            "sources": [c["source"] for c in chunks],
        }


rag = RAGPipeline(collection=collection, embed_model=embed_model, llm_client=llm_client)
print("RAGPipeline ready")

In [ ]:
# Exercise 4.1 solution — Generate and cache results
with open("../data/sample_qa.json") as f:
    qa_data = json.load(f)

CACHE_PATH = Path("../data/rag_results.json")

if CACHE_PATH.exists():
    with open(CACHE_PATH) as f:
        rag_results = json.load(f)
    print(f"Loaded {len(rag_results)} cached results")
else:
    rag_results = []
    print("Running RAG pipeline on all questions...")
    for qa in tqdm(qa_data["questions"]):
        result = rag.ask(qa["question"])
        result["ground_truth"] = qa["ground_truth"]
        result["id"] = qa["id"]
        rag_results.append(result)

    with open(CACHE_PATH, "w") as f:
        json.dump(rag_results, f, indent=2)
    print(f"Saved {len(rag_results)} results to {CACHE_PATH}")

In [ ]:
# Exercise 4.2 solution — Build EvaluationDataset
from ragas import EvaluationDataset

eval_data = [
    {
        "user_input": result["question"],
        "response": result["answer"],
        "retrieved_contexts": result["contexts"],
        "reference": result["ground_truth"],
    }
    for result in rag_results
]

dataset = EvaluationDataset.from_list(eval_data)
print(f"EvaluationDataset: {len(dataset)} examples")

In [ ]:
# Debug — raw LLM response for one evaluation call
import json as _json

r = rag_results[0]
contexts_text = "\n\n---\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(r["contexts"]))

_EVAL_PROMPT = """You are evaluating a RAG system. Score the response on all four dimensions.

Return ONLY a JSON object — no explanation, no markdown, just the JSON:
{{
  "faithfulness": <0.0-1.0>,
  "answer_relevancy": <0.0-1.0>,
  "context_precision": <0.0-1.0>,
  "context_recall": <0.0-1.0>
}}

Question: {question}
Ground truth: {ground_truth}
Retrieved contexts:
{contexts}
Answer: {answer}"""

prompt = _EVAL_PROMPT.format(
    question=r["question"],
    ground_truth=r.get("ground_truth", ""),
    contexts=contexts_text,
    answer=r["answer"],
)

try:
    resp = llm_client.chat.completions.create(
        model=FAST_MODEL, max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = resp.choices[0].message.content
    print("RAW RESPONSE:")
    print(repr(raw))
    print()
    print("PARSED:")
    print(raw)
except Exception as e:
    print(f"API ERROR: {type(e).__name__}: {e}")


In [ ]:
# Exercise 4.3 solution — Run evaluation
from src.evaluator import run_evaluation

print("Running evaluation (one LLM call per question)...")
df = run_evaluation(rag_results, llm_client=llm_client, judge_model=FAST_MODEL)

print("\nAggregate scores:")
print(df[["faithfulness", "answer_relevancy", "context_precision", "context_recall"]].mean().round(3))
df


In [ ]:
# Exercise 4.4 solution — Visualise and analyse
# Add question text
df["question_short"] = [r["question"][:40] + "..." for r in rag_results]

metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
means = df[metrics].mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar chart
colors = ["#e74c3c" if v < 0.6 else "#f39c12" if v < 0.75 else "#27ae60" for v in means]
axes[0].bar(means.index, means.values, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_ylim(0, 1)
axes[0].axhline(0.6, color="red", linestyle="--", alpha=0.5, label="Caution")
axes[0].axhline(0.75, color="orange", linestyle="--", alpha=0.5, label="Good")
axes[0].set_title("Mean RAGAS Scores")
axes[0].set_ylabel("Score (0–1)")
axes[0].legend(fontsize=8)
for i, (m, v) in enumerate(zip(means.index, means.values)):
    axes[0].text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)

# Heatmap
heat_data = df[metrics].copy()
heat_data.index = df["question_short"]
sns.heatmap(heat_data, ax=axes[1], cmap="RdYlGn", vmin=0, vmax=1,
            annot=True, fmt=".2f", linewidths=0.5)
axes[1].set_title("Per-Question RAGAS Scores")

plt.tight_layout()
plt.show()

print("\nBottom 3 by faithfulness:")
print(df.nsmallest(3, "faithfulness")[["question_short", "faithfulness"]].to_string(index=False))

print("\nBottom 3 by context recall:")
print(df.nsmallest(3, "context_recall")[["question_short", "context_recall"]].to_string(index=False))

---
## Part 5 — Alternative Evaluation with promptfoo

**promptfoo** is a CLI tool for testing and evaluating LLM applications.  
It differs from our direct evaluator in two key ways:

| | Direct LLM evaluator | promptfoo |
|---|---|---|
| Output | Numeric scores (0–1) | Pass / Fail + score |
| Config | Python code | YAML file (version-controlled) |
| Report | DataFrame / chart | Interactive HTML UI |
| CI/CD | Custom script | Built-in `--output json` |

We use the **pre-computed `rag_results`** from Part 1 so the pipeline doesn't re-run.  
An `echo` provider passes each answer straight to the LLM judge.


In [ ]:
# Exercise 4.7 solution — Generate promptfoo config
from src.evaluator import write_promptfoo_config

config_path = write_promptfoo_config(
    rag_results,
    output_path="../promptfooconfig.yaml",
    judge_model=FAST_MODEL,
)
print(f"Config written to: {config_path}")

# Inspect the first test case
import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)
print(f"\n{len(cfg['tests'])} test cases")
print("\nFirst test vars:")
first = cfg['tests'][0]['vars']
print(f"  question: {first['question'][:80]}")
print(f"  answer:   {first['answer'][:80]}...")

In [ ]:
# Exercise 4.8 solution — Run promptfoo
import subprocess, os
from dotenv import dotenv_values

# Pass .env values as env vars; map to OPENAI_* so promptfoo's grader picks them up
_dotenv = dotenv_values("../.env")
_env = {
    **os.environ,
    **_dotenv,
    "OPENAI_API_KEY": _dotenv.get("NETLIGHT_API_KEY", ""),
    "OPENAI_BASE_URL": _dotenv.get("NETLIGHT_BASE_URL", "").rstrip("/"),
}

proc = subprocess.run(
    ["npx", "--yes", "promptfoo@0.82.0", "eval",
     "--config", "../promptfooconfig.yaml",
     "--output", "../data/promptfoo_results.json"],
    capture_output=True, text=True, env=_env,
)
print(proc.stdout[-2000:] if len(proc.stdout) > 2000 else proc.stdout)
if proc.returncode != 0:
    print("stderr:", proc.stderr[-1000:])

In [ ]:
# Parse promptfoo results into a DataFrame
from pathlib import Path

pf_path = Path("../data/promptfoo_results.json")
if not pf_path.exists():
    print("promptfoo_results.json not found — run the eval cell above first.")
else:
    with open(pf_path) as f:
        pf_data = json.load(f)

    rows = []
    for r in pf_data.get("results", {}).get("results", []):
        rows.append({
            "question": r.get("vars", {}).get("question", "")[:60],
            "pass": r.get("gradingResult", {}).get("pass", False),
            "score": r.get("gradingResult", {}).get("score", 0.0),
            "reason": r.get("gradingResult", {}).get("reason", ""),
        })

    df_pf = pd.DataFrame(rows)
    print(f"Pass rate: {df_pf['pass'].mean():.1%}")
    display(df_pf[["question", "pass", "score"]])

> **View the promptfoo report in your terminal:**
> ```bash
> npx promptfoo@0.82.0 view
> ```
> Then open http://localhost:15500 in your browser.
